# Three samplers, and the question nobody asks until the numbers differ

A team runs NUTS through NumPyro. Someone else runs the "same" model in PyMC because it
handles a discrete latent better. The two posteriors disagree by more than Monte-Carlo noise,
and the next three days go into finding out which of the two model definitions was wrong —
because there were two model definitions. There always are, once a second backend exists.

`Backend` is a small protocol: `sample`, `optimize`, `laplace`, each taking a `ModelSpec` and
a data dict — **never a Python callable** (review A2). Every backend compiles the *same
spec*: `core.value` for Laplace, the jax interpreter for NumPyro, the pytensor interpreter
for PyMC. None of them is a second definition of the model, which is what makes the
disagreement above impossible rather than merely unlikely.

`get_backend("laplace")` is always available; `get_backend("numpyro")` returns a typed
`Unsupported` when the extra is missing.

In [ ]:
import sys

import numpy as np

from axiom.core import D, Data, Likelihood, ModelSpec, Param, Prior, is_failure
from axiom.infer import (
    BACKEND_NAMES, Backend, LaplaceBackend, NUTS_SAMPLERS, NumpyroBackend, NutsSampler,
    PointEstimate, Posterior, PymcBackend, SampleSettings, SamplerStatus, get_backend, samplers,
)

from axiom.display import enable, table

import sys as _sys; _sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, compare, intervals

enable();  # every axiom result renders itself from here on

In [ ]:
print(BACKEND_NAMES)
lap = get_backend("laplace")
print(isinstance(lap, Backend), getattr(lap, "name", lap))
npr = get_backend("numpyro")
print(npr if is_failure(npr) else npr.name)
pmc = get_backend("pymc")
print(pmc if is_failure(pmc) else pmc.name)
print("heavy modules loaded by importing axiom.infer:",
      [m for m in ("jax", "numpyro", "pymc", "pytensor", "arviz") if m in sys.modules])

That last line is the dependency budget holding: importing `axiom.infer` pulls in no sampler
at all. A backend's heavy import happens when you ask for it, and asking for one that is not
installed gets a typed refusal rather than an `ImportError` three frames down.

A conjugate normal–normal model to compare against the closed form — the one case where
"which of them is right" has an answer that does not depend on any of them.

In [ ]:
y = Data(name="y", dimension=D.outcome)
mu = Param(name="mu", dimension=D.outcome, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 10.0}))
sigma = Param(name="sigma", dimension=D.outcome, prior=Prior(family="fixed", hyper={"value": 1.0}))
model = ModelSpec(name="normal_normal", mean=mu, outcome=y, likelihood=Likelihood(family="normal", scale="sigma"), parameters=(mu, sigma))
rng = np.random.default_rng(0)
data = {"y": rng.normal(2.0, 1.0, 50)}
n, ybar = 50, data["y"].mean()
post_var = 1 / (1 / 100 + n)
print("closed form:", round(post_var * n * ybar, 4), "±", round(np.sqrt(post_var), 4))

In [ ]:
from axiom.display import show

backend = LaplaceBackend()
pe: PointEstimate = backend.optimize(model, data, seed=0)
print(pe.method, pe.converged, round(float(pe.theta["mu"]), 4))
post = backend.laplace(model, data, draws=4000, seed=0)
if isinstance(post, Posterior):
    s = post.summary("mu")
    print(round(s.mean, 4), round(s.sd, 4), post.provenance["hessian_pd"], post.provenance["seed"])
else:
    show(post)

In [ ]:
settings = SampleSettings(draws=300, tune=300, chains=2)
if not is_failure(npr):
    nuts = NumpyroBackend()
    mc = nuts.sample(model, data, draws=settings.draws, tune=settings.tune, chains=settings.chains, seed=0)
    if isinstance(mc, Posterior):
        print(round(mc.summary("mu").mean, 3), mc.provenance["divergences"], mc.provenance["method"])

## PyMC, and the four samplers behind it

The third backend. Like the NumPyro one it writes **no model**: the PyMC model is one
`pm.Flat` per free parameter in unconstrained space plus a single `pm.Potential` holding
`core.interpret.pytensor.compile_log_density`.

What PyMC adds is a **choice of NUTS implementation** over that one graph. `samplers()`
says which are installed; `samplers(probe=True)` actually runs two draws through each,
because an external sampler can import perfectly and still be out of step with the
installed PyMC. Installed is not the same as usable, and the report distinguishes them —
the alternative is a stack trace forty minutes into a fit.

In [ ]:
print("PyMC can dispatch NUTS to:", NUTS_SAMPLERS, "\n")
print(f"{'sampler':10s} {'installed':>10s}   probed")
rows = []
for name, status in samplers().items():
    probed: SamplerStatus = samplers(probe=True)[name] if status.usable else status
    rows.append([name, str(status.usable), str(probed)[:60]])
table(rows, headers=("sampler", "usable", "probed"))

`fit` and the backend protocol take a `Backend` *instance* as well as a name, which is
the seam for any backend-specific option — nothing PyMC-shaped leaks into a generic
signature.

In [ ]:
choice: NutsSampler = "pymc"
pymc_backend = PymcBackend(nuts_sampler=choice, target_accept=0.9)
assert isinstance(pymc_backend, Backend)
draws = pymc_backend.sample(model, data, draws=settings.draws, tune=settings.tune,
                            chains=settings.chains, seed=0)
assert isinstance(draws, Posterior)
print("mu:", round(draws.summary("mu").mean, 3), "+-", round(draws.summary("mu").sd, 3))
print("closed form:", round(post_var * n * ybar, 3), "+-", round(np.sqrt(post_var), 3))
print("\nprovenance:", {k: draws.provenance[k] for k in
      ("backend", "method", "nuts_sampler", "seed", "init", "init_per_chain", "divergences")})

### The samplers agree, because the graph is the same

The sampler is a speed choice, not a modelling one — so the only thing that should differ
between them is Monte Carlo noise. `tests/contracts/test_backend_equivalence.py` holds
every usable one to 0.15 posterior sd of PyMC's own.

In [ ]:
import time

usable = [name for name, status in samplers(probe=True).items() if status.usable]
rows, runs = [], {}
for name in usable:
    started = time.perf_counter()
    got = PymcBackend(nuts_sampler=name).sample(model, data, draws=500, tune=500, chains=2, seed=0)
    runs[f"pymc/{name}"] = (got.summary("mu"), time.perf_counter() - started)
    rows.append([f"pymc/{name}", f"{got.summary('mu').mean:.4f}", f"{got.summary('mu').sd:.4f}",
                 f"{runs[f'pymc/{name}'][1]:.1f}"])
table(rows, headers=("backend", "mean(mu)", "sd(mu)", "seconds"))
if not is_failure(npr):
    got = NumpyroBackend().sample(model, data, draws=500, tune=500, chains=2, seed=0)
    runs["numpyro"] = (got.summary("mu"), float("nan"))
    print(f"{'numpyro backend':18s} {got.summary('mu').mean:8.4f} {got.summary('mu').sd:8.4f}")
lap_post = LaplaceBackend().laplace(model, data, draws=4000, seed=0)
runs["laplace"] = (lap_post.summary("mu"), float("nan"))
print(f"{'closed form':18s} {post_var * n * ybar:8.4f} {np.sqrt(post_var):8.4f}")

In [ ]:
exact = post_var * n * ybar
fig = intervals(
    [(name, s.mean, s.mean - 2 * s.sd, s.mean + 2 * s.sd) for name, (s, _) in runs.items()],
    ref=exact, ref_label="closed form",
    title="Every route, against the answer",
    subtitle="posterior mean ±2 sd for mu — four samplers and a normal approximation of one ModelSpec",
    x_title="mu",
)
caption(fig, "Nothing here is a coincidence of tuning. The spread across routes is Monte-Carlo "
             "noise around a conjugate posterior that all of them compiled from the same spec, "
             "and the contract test fails the build if any route drifts past 0.15 sd.")

In [ ]:
timed = {name: secs for name, (_, secs) in runs.items() if secs == secs}
fig = compare(
    list(timed), list(timed.values()),
    highlight=min(timed, key=timed.get),
    value_fmt="{:.1f}s",
    title="…so the choice left is speed",
    subtitle="wall-clock for 2 chains × 500 draws on the same model and seed",
    x_title="seconds",
)
caption(fig, "This is the decision `samplers()` exists to inform. It is a real difference and "
             "it is nobody's modelling opinion.")

### `optimize` and `laplace` are not reimplemented

A backend is a *sampler*; the mode search and the normal approximation are numpy/scipy and
shared, so all three backends return the same answer for them — not agree to a tolerance,
the same answer, because it is the same code.

In [ ]:
mode: PointEstimate = pymc_backend.optimize(model, data, seed=0)
print("mode:", {k: round(float(v), 4) for k, v in mode.theta.items()}, "converged:", mode.converged)
approx = pymc_backend.laplace(model, data, draws=500, seed=0)
print("laplace via the pymc backend:", round(approx.summary("mu").mean, 4),
      "| method:", approx.provenance["method"])
print("same as LaplaceBackend:",
      round(LaplaceBackend().laplace(model, data, draws=500, seed=0).summary("mu").mean, 4))

## What this bought you

A model you can move between samplers to find out which is fastest, without ever finding out
which one you wrote down differently. Plus a package that imports in a fraction of a second
because none of them is loaded until you ask.